In [26]:
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np
import scipy
import scipy.signal as sig
import copy
import cv2 

In [27]:
colors = ("blue", "green", "red")
%matplotlib inline

In [ ]:
raw_data = pd.read_csv("percorso_PPG.csv", index_col=0)#inserire il .csv ottenuto con il codice PPG_to_BR
raw_data = raw_data.iloc[:,:]
raw_data.head()
video_path = "percorso_video.mp4"#inserire il percorso al video (serve solo per il calcolo del framerate)

In [29]:
def get_video_fps(video_path):
    # Apri il video
    cap = cv2.VideoCapture(video_path)

    # Controlla se il video è stato aperto correttamente
    if not cap.isOpened():
        print("Errore: impossibile aprire il video.")
        return None

    # Leggi il frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Frame rate del video: {fps:.2f} FPS")

    # Rilascia la risorsa
    cap.release()
    return fps

fr = get_video_fps(video_path)

fr = int(round(fr))
print(fr)

Frame rate del video: 30.01 FPS
30


In [30]:
n_samples = raw_data.shape[0]
t = np.arange(0, (n_samples)/fr, 1/fr)

In [31]:
def limit_signal(signal, threshold_high=None, threshold_low=None):
    """
    Limita i valori del segnale che superano le soglie specificate.
    Se threshold_high/low non sono specificati, usa 2 * std dev.
    """
    if threshold_high is None:
        threshold_high = np.mean(signal) + 2 * np.std(signal)
    if threshold_low is None:
        threshold_low = np.mean(signal) - 2 * np.std(signal)

    limited_signal = signal.copy()
    for i in range(len(limited_signal)):
        if limited_signal[i] > threshold_high:
            limited_signal[i] = threshold_high
        elif limited_signal[i] < threshold_low:
            limited_signal[i] = threshold_low
    return limited_signal

limited_sgn = raw_data.copy()

for i, col in enumerate(colors):

    # Estrae la colonna come array NumPy
    signal_array = raw_data.iloc[:, i].values

    # Applica la limitazione del segnale
    limited_array = limit_signal(signal_array)
    signal_min = np.min(limited_array)
    signal_max = np.max(limited_array) 
    normalized_signal = -1 + ((limited_array - signal_min ) * (2)) / (signal_max - signal_min)
    signal_avg = np.mean(normalized_signal)
    normalized_signal = normalized_signal - signal_avg


    # Salva il segnale limitato
    limited_sgn.iloc[:, i] = normalized_signal


In [32]:
crossings_salita = [[],[],[]]
crossings_discesa = [[],[],[]]
for i, col in enumerate(colors):
    signal = np.asarray(limited_sgn.iloc[:, i])
    for j in range(0, len(signal)-1):
        if(signal[j]<0 and signal[j+1]>0):
            crossings_salita[i].append(j)#zero crossing in salita
        elif(signal[j]>0 and signal[j+1]<0):#zero crossing in discesa
            crossings_discesa[i].append(j)

In [33]:
BR_salita = np.zeros(3)
BR_discesa = np.zeros(3)
BR_media = np.zeros(3)
for i, col in enumerate(colors):
    BR_salita[i] = (len(crossings_salita[i])-1)/(crossings_salita[i][-1]-crossings_salita[i][0])*fr*60
    BR_discesa[i] = (len(crossings_discesa[i])-1)/(crossings_discesa[i][-1]-crossings_discesa[i][0])*fr*60
    BR_media[i] = (BR_salita[i]+BR_discesa[i])/2

In [34]:
print(f"canale\t\tblu\tverde\trosso".replace('.', ','))
print(f"BR salita\t{round(BR_salita[0],1)}\t{round(BR_salita[1],1)}\t{round(BR_salita[2],1)}".replace('.', ','))
print(f"BR discesa\t{round(BR_discesa[0],1)}\t{round(BR_discesa[1],1)}\t{round(BR_discesa[2],1)}".replace('.', ','))
print(f"BR media\t{round(BR_media[0],1)}\t{round(BR_media[1],1)}\t{round(BR_media[2],1)}".replace('.', ','))

canale		blu	verde	rosso
BR salita	9,8	12,4	10,4
BR discesa	9,7	12,5	10,4
BR media	9,7	12,5	10,4
